## FORECASTING EXCHANGE RATES USING TIME SERIES ANALYSIS

```
Objective:
Leverage ARIMA and Exponential Smoothing techniques to forecast future exchange rates based on historical data provided in the exchange_rate.csv dataset. 

Dataset:
The dataset contains historical exchange rate with each column representing a different currency rate over time. The first column indicates the date, and the second column represents exchange rates USD to Australian Dollar.

Part 1: Data Preparation and Exploration
1.	Data Loading: Load the exchange_rate.csv dataset and parse the date column appropriately.
2.	Initial Exploration: Plot the time series for currency to understand their trends, seasonality, and any anomalies.
3.	Data Preprocessing: Handle any missing values or anomalies identified during the exploration phase.

Part 2: Model Building - ARIMA
1.	Parameter Selection for ARIMA: Utilize ACF and PACF plots to estimate initial parameters (p, d, q) for the ARIMA model for one or more currency time series.
2.	Model Fitting: Fit the ARIMA model with the selected parameters to the preprocessed time series.
3.	Diagnostics: Analyze the residuals to ensure there are no patterns that might indicate model inadequacies.
4.	Forecasting: Perform out-of-sample forecasting and visualize the predicted values against the actual values.

Part 3: Model Building - Exponential Smoothing
1.	Model Selection: Depending on the time series characteristics, choose an appropriate Exponential Smoothing model (Simple, Holt’s Linear, or Holt-Winters).
2.	Parameter Optimization: Use techniques such as grid search or AIC to find the optimal parameters for the smoothing levels and components.
3.	Model Fitting and Forecasting: Fit the chosen Exponential Smoothing model and forecast future values. Compare these forecasts visually with the actual data.

Part 4: Evaluation and Comparison
1.	Compute Error Metrics: Use metrics such as MAE, RMSE, and MAPE to evaluate the forecasts from both models.
2.	Model Comparison: Discuss the performance, advantages, and limitations of each model based on the observed results and error metrics.
3.	Conclusion: Summarize the findings and provide insights on which model(s) yielded the best performance for forecasting exchange rates in this dataset.

Deliverables:
●	Include visualizations and explanations for the choices and findings at each step.
●	Well-commented Python code that used to conduct the analysis and build the models.

Assessment Criteria:
●	Accuracy and completeness of the data preparation and exploration steps.
●	Justification for model selection and parameter tuning decisions.
●	Clarity and depth of the analysis in the diagnostics and model evaluation stages.
This assignment offers hands-on experience with real-world data, applying sophisticated time series forecasting methods to predict future currency exchange rates.

```

## Answers:
### Part 1: Data Preparation and Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Uploading the dataset to our enviroment
dataset = pd.read_csv("exchange_rate.csv", parse_dates=['date'], dayfirst=True)

# Working on a copy 
df = dataset.copy()

# Understanding data set
print("\n<----------INFO----------->\n")
print(df.info())

print("\n<-----------DESCRIBE ONLY NUMERICAL---------->\n")
print(df.describe())

print("\n<---------DESCRIBE ALL NUMERICAL AND CATEGORICAL--------->\n")
print(df.describe(include='all'))

print("\n<---------MISSING VALUES--------->\n")
print(df.isnull().sum())

In [ ]:
# Plot time and series
import matplotlib.dates as mdates
plt.figure(figsize=(12,5))
plt.plot(df["date"],df['Ex_rate'],label="Exchange rate (USD to AUD)")
plt.title("USD to AUD Exchange rate over time")
plt.xlabel("Date")
plt.ylabel("Exchange rate")

plt.gca().xaxis.set_major_locator(mdates.YearLocator(base=2))   
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
# ADF Testing
# Checking whether model is stationary or not using hypothesis testing 
from statsmodels.tsa.stattools import adfuller

result = adfuller(df['Ex_rate'])
print("ADF statistics :",result[0])
print("p-value is given by :",result[1])
print("Critical value is given by:",result[4])

if result[1] < 0.05:
    print("Series is stattionsry, no differencing is needed (d=0)")
else:
    print("Series is non-stationary, differencing is neede (d=1)")

## Model Building -ARIMA

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
import statsmodels.api as sm

# plot ACF and PACF to guess p, q
plot_acf(df['Ex_rate'])
plot_pacf(df['Ex_rate'])
plt.show()

In [ ]:
# ARIMA model 
model = ARIMA(df["Ex_rate"], order=(1,1,1))
arima_result = model.fit()

print(arima_result.summary())

In [ ]:
model = ARIMA(df["Ex_rate"], order=(1,1,0))
arima_result = model.fit()

print(arima_result.summary())

In [ ]:
# In between order(1,1,1) and order(1,1,0) we sill chose the second one cause in case of (1,1,1) ml l1 p value >0.05
# ARIMA model 
model = ARIMA(df["Ex_rate"], order=(1,1,0))
arima_result = model.fit()

print(arima_result.summary())

# Residual digonistics
arima_result.plot_diagnostics(figsize=(15,10))

In [ ]:
# Forecasting next 30 days 
forecast_arima = arima_result.get_forecast(steps=30)
forecast_df = forecast_arima.conf_int()
forecast_df['Forecast']= forecast_arima.predicted_mean


plt.figure(figsize=(12,5))
plt.plot(df['Ex_rate'].iloc[-200:], label="Recent Historical")
plt.plot(forecast_df['Forecast'],label = "ARIMAForecast",color = 'red')
plt.fill_between(forecast_df.index,
                 forecast_df.iloc[:,0],
                 forecast_df.iloc[:,1], color="pink", alpha=0.3)
plt.legend()
plt.show()

## Part 3: Model Building - Exponential Smoothing

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import itertools



# Defining possible parameter values 
trend_options = [None,'add','mul']
seasonal_options = [None,'add','mul']
seasonal_periods = [None,7,12,30] # Weekly ,monthly and so on

best_aic = float('inf')
best_params = None
best_model = None

for trend,seasonal,sp in itertools.product(trend_options, seasonal_options,seasonal_periods):
    try:
        hw_model= ExponentialSmoothing(df['Ex_rate'], trend=trend,seasonal=seasonal,seasonal_periods=sp).fit()
        if hw_model.aic < best_aic:
            best_aic = hw_model.aic
            best_params = (trend,seasonal,sp)
            best_model = hw_model
    except:
        continue

print("Best parameters:",best_params)
print("Best AIC:",best_aic)

In [ ]:
# Forecasting the next 30 days
forecast_hw = hw_model.forecast(30)


plt.figure(figsize=(12,5))
plt.plot(df["Ex_rate"].iloc[-100:],label = "Actual")
plt.plot(forecast_hw,label="Exponential Smothing Forecast",color="green")
plt.legend()
plt.show()

## Part4: Evaluation and Compairision

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# In-sample predications
df['ARIMA_Pred'] = arima_result.fittedvalues
df['hw_Pred'] = hw_model.fittedvalues

# Compute error metrics
mae_arima = mean_absolute_error(df['Ex_rate'],df['ARIMA_Pred'])
mse_arima = mean_squared_error(df['Ex_rate'],df['ARIMA_Pred'])

mae_hw = mean_absolute_error(df['Ex_rate'],df['hw_Pred'])
mse_hw = mean_squared_error(df['Ex_rate'],df['hw_Pred'])

print("ARIMA------> MAE:",mae_arima, "\tMSE:",mse_arima)
print("HW------> MAE:",mae_hw, "\tMSE:",mse_hw)

## Chossing model:
```
i.From the above result,we can clearly see that : ARIMA_MSE < HW_MSE
ii. And also the ARIMA_MAE approximately same to the HW_AME
        Hence, As the mse is less in case of the ARIMA model so, we are going to choose ARIMA model instead of the HW model.
```